In [1]:
import os
import faiss
import time
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline


/home/rgktongole/.local/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-01-21 13:57:54.872118: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-21 13:57:57.336657: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-21 13:58:03.830747: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [4]:
def load_documents(folder="documents"):
    texts = []
    for file in os.listdir(folder):
        if file.endswith(".txt"):
            with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
                texts.extend(f.read().split("\n"))
    return [t for t in texts if t.strip()]

docs = load_documents("/home/rgktongole/Desktop/rag_chatbot/documents")
print("Documents loaded:", len(docs))


Documents loaded: 5


In [5]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embed_model.encode(docs)


In [6]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))


In [8]:
pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_new_tokens=150
)



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [9]:
def retrieve(query, top_k=3):
    query_emb = embed_model.encode([query])
    distances, indices = index.search(query_emb, top_k)
    return [docs[i] for i in indices[0]]


In [10]:
def generate_answer(query):
    start = time.time()
    context = "\n".join(retrieve(query))

    prompt = f"""
Answer the question ONLY using the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{query}
"""

    response = llm(prompt)[0]["generated_text"]
    return response, time.time() - start


In [11]:
ans, latency = generate_answer("What is FAISS?")
print(ans)
print("Latency:", latency)


a vector database library for efficient similarity search
Latency: 55.09889268875122
